# Task 2 — Zero-shot CoT với Qwen2.5-Coder-7B-Instruct (Kaggle)

Prompt vai giảng viên, chấm theo 7 bước rồi xuất dòng `KẾT LUẬN: {"taxonomy_error": [...]}` (xem `docs/task2_prompting.md`).
Không train: chấm trên cả 32 bài gold, rồi (tuỳ chọn) dự đoán 360 bài public.

1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Dán file id Google Drive của bộ dữ liệu vào `DRIVE_ID` (không commit) hoặc dùng secret `PUBLIC_TEST_DRIVE_ID`. Tuỳ chọn `GITHUB_TOKEN` nếu repo private, `HF_TOKEN`.
3. Sửa `BRANCH` cho đúng nhánh đã push code CoT.
4. Run All.

**Silver (tập sinh, để chỉnh prompt):** đặt `SILVER = 'silver'` (462 bài) hoặc `'heldout'` (65 bài, 4 đề chưa thấy) và `SILVER_DRIVE_ID` (hoặc secret `SILVER_DRIVE_ID`, file `generate-dataset.zip`). Notebook tải về `data/synth/v1/generate-dataset/` bằng `scripts/fetch_synth_data.py` rồi chấm thêm tập đó (thư mục kết quả có hậu tố `_silver` / `_silver_heldout`). Chỉnh prompt theo số trên silver, không theo gold.

**Output** (`/kaggle/working`):
- `task2_predictions.json` (nộp bài, nếu `PREDICT = True`)
- `cot_gold/`, `cot_public/`: `report.html` (mở trên máy để debug: điểm từng nhãn, TP/FP/FN, lời chấm từng bài), `debug.jsonl`, `run.log`, `raw_outputs.jsonl`, `metrics.json`, `resources.json`
- `task2_cot_bundle/`: bundle để chạy lại sau bằng `scripts/predict_task2_cot.py --bundle ...` (`SAVE_WEIGHTS = True` thì kèm trọng số ~15 GB)

**Thời gian ước tính:** T4, fp16, greedy ~1024 token ra → khoảng 40–60 giây/bài; 32 gold ≈ 30 phút, 360 public ≈ 4–6 giờ. Kaggle giới hạn 12 giờ/phiên; chạy lại sẽ dùng cache trong `WORK` nếu chưa mất phiên.


In [ ]:
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2-zeroshot'# nhánh chứa scripts/run_task2_cot.py
DRIVE_ID = ''             # paste the Google Drive file id here on Kaggle (do not commit it: the repo is public)
WORK = '/kaggle/tmp'      # student data + repo live here, not in /kaggle/working (which is saved with the notebook)
COMPILE_SOURCE = 'self'   # 'dataset' if DRIVE_ID points to the compiled zip (compile_log already filled)
PREDICT = True            # also label the 360 public samples
LIMIT = None              # e.g. 4 for a quick check on the first gold samples (no public run)
LOAD_IN_4BIT = False      # True if the GPU has < 16 GB in total
SAVE_WEIGHTS = False      # True: copy the ~15 GB fp16 weights into the bundle (fits /kaggle/working's 20 GB, barely)
SILVER_DRIVE_ID = ''      # Drive file id of generate-dataset.zip (or secret SILVER_DRIVE_ID); do not commit
SILVER = ''               # '' = off | 'silver' (462 synthetic, prompt tuning) | 'heldout' (65, 4 unseen exams)
SILVER_LIMIT = None       # e.g. 100: only the first N silver samples


In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed — check REPO/BRANCH/GITHUB_TOKEN') from None  # never print the token URL
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)  # drop token from .git/config
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q gdown 'transformers>=4.51,<5' accelerate safetensors scikit-learn pyyaml bitsandbytes
!g++ --version


In [ ]:
import gdown
drive_id = DRIVE_ID or secret('PUBLIC_TEST_DRIVE_ID')
assert drive_id, 'Set DRIVE_ID in the config cell (or add the PUBLIC_TEST_DRIVE_ID secret)'
os.makedirs(f'{WORK}/download', exist_ok=True)
path = gdown.download(id=drive_id, output=f'{WORK}/download/data.zip', quiet=False)
assert path is not None, 'gdown failed — check PUBLIC_TEST_DRIVE_ID and sharing permissions'
!python scripts/unpack_data.py {WORK}/download --dest data/raw
!ls data/raw

In [ ]:
# Synthetic silver set -> data/synth/v1/generate-dataset/ (+ heldout/), only when SILVER is set
if SILVER:
    os.environ['SILVER_DRIVE_ID'] = SILVER_DRIVE_ID or secret('SILVER_DRIVE_ID')
    assert os.environ['SILVER_DRIVE_ID'], 'Set SILVER_DRIVE_ID in the config cell (or add the SILVER_DRIVE_ID secret)'
    !python scripts/fetch_synth_data.py


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import yaml
run_cfg = yaml.safe_load(open('configs/task2/run_cot.yaml'))
run_cfg['hf']['load_in_4bit'] = LOAD_IN_4BIT
yaml.safe_dump(run_cfg, open(f'{WORK}/run_cot.yaml', 'w'), allow_unicode=True)
OUT = f'{WORK}/outputs'
CMD = ['python', 'scripts/run_task2_cot.py', '--run-config', f'{WORK}/run_cot.yaml', '--backend', 'hf',
       '--compile-source', COMPILE_SOURCE, '--output-dir', OUT, '--export-bundle', '/kaggle/working/task2_cot_bundle']
if LIMIT:
    CMD += ['--limit', str(LIMIT)]
elif PREDICT:
    CMD += ['--predict']
if SAVE_WEIGHTS:
    CMD += ['--save-weights']
print(' '.join(CMD))


In [ ]:
# Streams progress (one line per sample: tokens in/out, seconds, tok/s). Everything is also in run.log.
!{' '.join(CMD)}


In [ ]:
import glob, shutil
# Score the silver set (prompt tuning). Run folder suffix: _silver / _silver_heldout
if SILVER:
    SILVER_FLAG = '--silver' if SILVER == 'silver' else '--silver-heldout'
    SBASE = ['--run-config', f'{WORK}/run_cot.yaml', '--backend', 'hf', '--compile-source', COMPILE_SOURCE,
             '--output-dir', OUT, SILVER_FLAG] + (['--limit', str(SILVER_LIMIT)] if SILVER_LIMIT else [])
    S1 = ' '.join(['python', 'scripts/run_task2_cot.py', *SBASE])
    print(S1)
    !{S1}
    for d in glob.glob(f'{OUT}/task2_cot_zero_shot_silver*'):
        dst = f'/kaggle/working/cot_{os.path.basename(d).split("__")[0].replace("task2_cot_zero_shot_", "")}'
        os.makedirs(dst, exist_ok=True)
        for name in ('report.html', 'debug.jsonl', 'run.log', 'metrics.json', 'resources.json', 'raw_outputs.jsonl'):
            if os.path.exists(f'{d}/{name}'):
                shutil.copy(f'{d}/{name}', dst)
        print('->', dst)


In [ ]:
import glob, json, os, shutil
from src.task2.evaluator import validate_predictions
gold_dir = [d for d in glob.glob(f'{OUT}/task2_cot_zero_shot__hf__*') if not d.endswith('_predict')][0]
KEEP = ['report.html', 'debug.jsonl', 'run.log', 'raw_outputs.jsonl', 'usage.jsonl', 'metrics.json',
        'resources.json', 'config.json', 'predictions.json', 'predictions_full.json']
def keep(src, name):
    dst = f'/kaggle/working/{name}'
    os.makedirs(dst, exist_ok=True)
    for f in KEEP:
        if os.path.exists(f'{src}/{f}'):
            shutil.copy(f'{src}/{f}', dst)
    return dst
keep(gold_dir, 'cot_gold')
print(json.dumps(json.load(open(f'{gold_dir}/metrics.json'))['systems'].get('api', {}), ensure_ascii=False, indent=1)[:1500])
pred_dirs = glob.glob(f'{OUT}/task2_cot_zero_shot__hf__*_predict')
if pred_dirs:
    keep(pred_dirs[0], 'cot_public')
    shutil.copy(f'{pred_dirs[0]}/predictions.json', '/kaggle/working/task2_predictions.json')
    sub = json.load(open('/kaggle/working/task2_predictions.json'))
    print('task2 public', len(sub), 'errors:', validate_predictions(sub))


In [ ]:
# Per-label view of the gold run (same numbers as report.html)
import pandas as pd
m = json.load(open(f'{gold_dir}/metrics.json'))['systems']['api']
df = pd.DataFrame(m['per_label']).T[['support', 'n_pred', 'precision', 'recall', 'f1']]
print(f"macro-F1={m['macro_f1']:.3f}  micro-F1={m['micro_f1']:.3f}  exact={m['exact_match']:.3f}")
df


In [ ]:
# Interactive debug report inline (also saved as /kaggle/working/cot_gold/report.html)
import html as _html
from IPython.display import HTML, display
page = open(f'{gold_dir}/report.html', encoding='utf-8').read()
display(HTML(f'<iframe srcdoc="{_html.escape(page)}" style="width:100%;height:900px;border:0"></iframe>'))
